In [1]:
import duckdb
import pandas as pd

# -----------------------------
# CONNECT TO DB
# -----------------------------
conn = duckdb.connect(
    "../database/airport_ops.duckdb"
)

# -----------------------------
# LOAD DATA
# -----------------------------
delay_df = conn.sql("""

SELECT

    airport_name,
    municipality,
    iso_country,

    nearby_flights,

    temperature_c,
    wind_kph,
    visibility_km,
    weather_condition,

    airport_stress_score

FROM airport_metrics

WHERE temperature_c
IS NOT NULL

""").df()

# -----------------------------
# RISK SCORE
# -----------------------------
delay_df["delay_risk_score"] = 0

# Wind
delay_df.loc[
    delay_df["wind_kph"] > 35,
    "delay_risk_score"
] += 25

# Visibility
delay_df.loc[
    delay_df["visibility_km"] < 5,
    "delay_risk_score"
] += 25

# Congestion
delay_df.loc[
    delay_df["nearby_flights"] > 100,
    "delay_risk_score"
] += 20

# Weather condition
bad_weather = [
    "Rain",
    "Heavy rain",
    "Storm",
    "Thunderstorm",
    "Snow",
    "Fog",
    "Mist"
]

delay_df.loc[
    delay_df[
        "weather_condition"
    ].isin(bad_weather),
    "delay_risk_score"
] += 30

# Stress score
delay_df.loc[
    delay_df[
        "airport_stress_score"
    ] > 70,
    "delay_risk_score"
] += 20

# Cap at 100
delay_df[
    "delay_probability"
] = delay_df[
    "delay_risk_score"
].clip(upper=100)

# -----------------------------
# RISK LEVEL
# -----------------------------
delay_df["delay_risk_level"] = pd.cut(

    delay_df[
        "delay_probability"
    ],

    bins=[-1,20,50,80,100],

    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

# -----------------------------
# MAIN CAUSE
# -----------------------------
def get_reason(row):

    if row["visibility_km"] < 5:
        return "Low Visibility"

    elif row["wind_kph"] > 35:
        return "Strong Wind"

    elif row["nearby_flights"] > 100:
        return "Airport Congestion"

    else:
        return "Normal Operations"

delay_df[
    "likely_delay_reason"
] = delay_df.apply(
    get_reason,
    axis=1
)

# -----------------------------
# SAVE TABLE
# -----------------------------
conn.register(
    "delay_df",
    delay_df
)

conn.execute("""

CREATE OR REPLACE TABLE
delay_prediction AS

SELECT *
FROM delay_df

""")

print(
    "Delay prediction created!"
)

delay_df.head()

conn.close()

Delay prediction created!
